# Training an A2C racing agent with GAE

This notebook develops the project-owned synchronous Advantage Actor-Critic agent and its Generalized Advantage Estimation targets. It uses the readable loop in `training/engines/a2c.py`, so rollout collection, bootstrapping, and the separate actor/critic updates remain visible.

The exact equations and episode-boundary rules come from [`docs/LEARNING.md`](../docs/LEARNING.md). The racing MDP and procedural circuits are specified in [`docs/MDP.md`](../docs/MDP.md) and [`docs/TRACK.md`](../docs/TRACK.md).

## Why move from REINFORCE to A2C?

REINFORCE weights each action log-probability with a complete Monte Carlo return. That estimator is direct but noisy, and the actor cannot update until every trajectory ends. A2C adds a critic \(v_w(O_t)\) that predicts the current policy's discounted value. Subtracting this learned state-dependent baseline asks a lower-variance question: was the sampled action better or worse than the policy normally does from this observation?

For each transition, A2C forms the temporal-difference residual

\[
\delta_t = R_{t+1} + \gamma B_t - v_w(O_t),
\]

where \(B_t=0\) after a true terminal state and otherwise is the critic prediction for the next observation. A time-limit truncation therefore retains one bootstrap value: the artificial episode cap is not treated as a physical terminal state.

## Generalized Advantage Estimation and the two losses

A one-step TD residual has lower variance than a Monte Carlo return but can be biased by an inaccurate critic. GAE exposes the trade-off through \(\lambda\):

\[
\widehat{\mathbb A}_t
= \delta_t + \gamma\lambda\widehat{\mathbb A}_{t+1}.
\]

The recursion stops at termination, truncation, environment-column changes, and rollout boundaries. The raw advantage creates the detached critic target \(y_t=\widehat{\mathbb A}_t+v_w(O_t)\); a standardized copy \(\widetilde{\mathbb A}_t\) is used only by the actor. For rollout \(B\),

\[
\mathcal L_{\mathrm{actor}}(\theta)
= -\frac{1}{|B|}\sum_{t\in B}
\log\pi_\theta(A_t\mid O_t)\,
\operatorname{detach}(\widetilde{\mathbb A}_t),
\]

\[
\mathcal L_{\mathrm{critic}}(w)
= \frac{1}{2|B|}\sum_{t\in B}
\left(v_w(O_t)-y_t\right)^2.
\]

A2C can update from fixed-length rollouts that cross episode boundaries, making it more sample-responsive than REINFORCE. Its remaining weakness is that each rollout is used for only one actor/critic update. PPO will reuse the same rollout for several minibatch epochs while constraining how far the actor moves.

## Configuration

The primary stopping rule is `TRAINING_INTERACTION_BUDGET`. Use the same value in all three notebooks for a fair environment-data comparison; the default below matches the frozen Experiment 1 planning budget.

Set `TRAIN_ON_RANDOM_CIRCUITS = True` to derive a fresh procedural circuit from every logical training episode index. The same root and episode index reproduce the same circuit in REINFORCE, A2C, and PPO. Rendering and greedy evaluation stay on one fixed reference circuit.

The initial dashboard shows 20 actions; the final dashboard allows the full 1,000-step environment limit.

In [ ]:
import sys
import warnings
from dataclasses import asdict, replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
warnings.filterwarnings(
    "ignore",
    message="pkg_resources is deprecated as an API.*",
    category=UserWarning,
    module="pygame.pkgdata",
)

from agents import A2CAgent
from configs import (
    FIXED_CRITIC_CONFIG,
    MEDIUM_ACTOR_CONFIG,
    A2CConfig,
    EnvironmentConfig,
    ExecutionConfig,
    ObservationNormalizationConfig,
)
from envs.racing import RacingEnv
from envs.tracks import TrackWithGeometry
from recording import RunCategory
from training.educational_visualization import (
    plot_algorithm_diagnostics,
    plot_driving_behavior,
    plot_optimization_and_exploration,
    plot_progress_and_efficiency,
    plot_task_performance,
    watch_deterministic_rollout,
)
from training.engines.a2c import A2CTrainingEngine
from training.evaluation import evaluate_deterministic
from training.normalization import RunningObservationNormalizer
from utils.random import (
    RunSeedStreams,
    SeedNamespace,
    SeedStream,
    configure_torch_determinism,
)

# Reproducibility and execution
ROOT_SEED = 0
DEVICE = "cpu"
TRAINING_INTERACTION_BUDGET = 2_000_000
NUM_ENVS = ExecutionConfig().environment_workers

# Circuit schedule
TRAIN_ON_RANDOM_CIRCUITS = False
SINGLE_CIRCUIT_SEED = 0

# Actor, critic, and A2C
ACTOR_CONFIG = replace(MEDIUM_ACTOR_CONFIG, learning_rate=3e-4)
CRITIC_LEARNING_RATE = 1e-3
A2C_CONFIG = A2CConfig(
    discount=0.9995,
    gae_lambda=0.95,
    transitions_per_rollout=2_048,
)
ENVIRONMENT_CONFIG = EnvironmentConfig()
NORMALIZATION_CONFIG = ObservationNormalizationConfig()

# Greedy evaluation and plots
EVALUATION_INTERVAL_INTERACTIONS = 50_000
EVALUATION_EPISODES = 16
MOVING_AVERAGE_WINDOW = 20

# Live rollout dashboard
RENDER_RESET_SEED = 123
VIEWER_MODE = "inline"
INITIAL_RENDER_STEPS = 20
FINAL_RENDER_STEPS = ENVIRONMENT_CONFIG.simulation.max_episode_steps
LIVE_FRAME_DELAY = 0.01

torch.cuda.is_available()

## Build the reference circuit, random streams, actor, critic, and engine

A2C adds an independent critic-initialization stream. Policy sampling, environment resets, circuit generation, and deterministic evaluation remain isolated. In random-circuit mode, circuit identity is a function of the logical episode number rather than the worker that happens to finish first.

In [ ]:
execution_config = ExecutionConfig(device=DEVICE, environment_workers=NUM_ENVS)
configure_torch_determinism(execution_config)
streams = RunSeedStreams(SeedNamespace.MULTI_CIRCUIT_DEVELOPMENT, ROOT_SEED)
track_streams = RunSeedStreams(
    SeedNamespace.EXPERIMENT_2_TRAINING_TRACK,
    ROOT_SEED,
)

reference_track = TrackWithGeometry.generate(
    SINGLE_CIRCUIT_SEED,
    track_config=ENVIRONMENT_CONFIG.track,
    vehicle_config=ENVIRONMENT_CONFIG.vehicle,
)
tracks = (reference_track,)


def training_track_seed(episode_index):
    generator = track_streams.get_numpy_generator(
        SeedStream.TRACK_GENERATION,
        substream_identity=episode_index,
    )
    return int(generator.integers(0, 2**32, dtype=np.uint32))


probe_environment = RacingEnv(reference_track, config=ENVIRONMENT_CONFIG)
observation_dimensions = probe_environment.observation_space.shape[0]
probe_environment.close()

agent = A2CAgent(
    observation_dimensions=observation_dimensions,
    actor_config=ACTOR_CONFIG,
    critic_config=FIXED_CRITIC_CONFIG,
    config=A2C_CONFIG,
    critic_learning_rate=CRITIC_LEARNING_RATE,
    actor_initialization_generator=streams.get_torch_generator(
        SeedStream.ACTOR_INITIALIZATION,
        device=DEVICE,
    ),
    critic_initialization_generator=streams.get_torch_generator(
        SeedStream.CRITIC_INITIALIZATION,
        device=DEVICE,
    ),
    sampling_generator=tuple(
        streams.get_torch_generator(
            SeedStream.POLICY_ACTION_SAMPLING,
            device=DEVICE,
            substream_identity=index,
        )
        for index in range(NUM_ENVS)
    ),
    device=DEVICE,
)
normalizer = RunningObservationNormalizer(
    observation_dimensions,
    NORMALIZATION_CONFIG,
)
engine = A2CTrainingEngine(
    agent=agent,
    tracks=tracks,
    environment_config=ENVIRONMENT_CONFIG,
    normalizer=normalizer,
    environment_reset_generator=tuple(
        streams.get_numpy_generator(
            SeedStream.ENVIRONMENT_RESETS,
            substream_identity=index,
        )
        for index in range(NUM_ENVS)
    ),
    track_selection_generator=tuple(
        streams.get_numpy_generator(
            SeedStream.TRAINING_TRACK_SELECTION,
            substream_identity=index,
        )
        for index in range(NUM_ENVS)
    ),
    track_seed_for_episode=(training_track_seed if TRAIN_ON_RANDOM_CIRCUITS else None),
    execution_config=execution_config,
)
evaluation_generator = streams.get_numpy_generator(SeedStream.EVALUATION)

training_mode = (
    "fresh procedural circuit per episode"
    if TRAIN_ON_RANDOM_CIRCUITS
    else f"fixed circuit seed {SINGLE_CIRCUIT_SEED}"
)
print(f"Training mode: {training_mode}")
print(f"Interaction budget: {TRAINING_INTERACTION_BUDGET:,}")
print(f"Actor parameters: {agent.actor_parameter_count:,}")
print(f"Critic parameters: {agent.critic_parameter_count:,}")

## Policy before training

The short dashboard uses deterministic bounded mean actions and frozen normalization, so it does not consume the stochastic A2C training stream. It combines the circuit view with current progress, return, speed, throttle, steering, and their histories.

In [ ]:
initial_rollout = watch_deterministic_rollout(
    "A2C policy before training",
    reference_track,
    agent,
    normalizer,
    ENVIRONMENT_CONFIG,
    max_steps=INITIAL_RENDER_STEPS,
    reset_seed=RENDER_RESET_SEED,
    viewer_mode=VIEWER_MODE,
    frame_delay=LIVE_FRAME_DELAY,
)
initial_rollout

## Scheduled greedy evaluation

Each checkpoint aggregates `EVALUATION_EPISODES` deterministic episodes. Return and progress means are plotted against the actual training-interaction counter, with plus/minus one population standard deviation. On the current deterministic fixed circuit that deviation is normally zero; keeping it explicit makes the presentation correct when evaluation later includes varied starts or circuits.

In [ ]:
evaluation_rows = []
evaluation_state = {
    "interactions": 0,
    "episodes": 0,
    "next_boundary": EVALUATION_INTERVAL_INTERACTIONS,
}


def run_greedy_evaluation(completed_training_episodes, training_interactions):
    returns = []
    progress_values = []
    for _ in range(EVALUATION_EPISODES):
        result = evaluate_deterministic(
            lambda: RacingEnv(reference_track, config=ENVIRONMENT_CONFIG),
            agent,
            normalizer,
            run_category=RunCategory.REDUCED_VALIDATION,
            evaluation_index=evaluation_state["episodes"],
            training_interactions=training_interactions,
            evaluation_interactions_before=evaluation_state["interactions"],
            reset_seed=int(evaluation_generator.integers(0, 2**32, dtype=np.uint32)),
            root_identity=ROOT_SEED,
            circuit_identity=str(reference_track.track.generation.seed),
        )
        episode = result.record.episode
        evaluation_state["episodes"] += 1
        evaluation_state["interactions"] = result.record.evaluation_interactions
        returns.append(episode.undiscounted_return)
        progress_values.append(episode.maximum_progress)

    evaluation_rows.append(
        {
            "completed_training_episodes": completed_training_episodes,
            "training_interactions": training_interactions,
            "return_mean": float(np.mean(returns)),
            "return_standard_deviation": float(np.std(returns)),
            "maximum_progress_mean": float(np.mean(progress_values)),
            "maximum_progress_standard_deviation": float(np.std(progress_values)),
        }
    )


def evaluate_on_schedule(episode_record, current_history):
    del episode_record
    while current_history.training_interactions >= evaluation_state["next_boundary"]:
        run_greedy_evaluation(
            len(current_history.episodes),
            current_history.training_interactions,
        )
        evaluation_state["next_boundary"] += EVALUATION_INTERVAL_INTERACTIONS


run_greedy_evaluation(0, 0)
evaluation_rows[-1]

## Train to the common interaction budget

The persistent workers step synchronously into a pooled time-by-environment rollout. GAE recurses only within each worker column. Full 2,048-transition rollouts produce normal A2C updates; the exact budget boundary produces one final shorter rollout rather than overshooting or discarding paid-for interactions.

In [ ]:
try:
    history = engine.train(
        TRAINING_INTERACTION_BUDGET,
        on_episode_end=evaluate_on_schedule,
    )
finally:
    engine.close()

if evaluation_rows[-1]["training_interactions"] != history.training_interactions:
    run_greedy_evaluation(
        len(history.episodes),
        history.training_interactions,
    )

print(f"Episodes completed: {len(history.episodes):,}")
print(f"Training interactions: {history.training_interactions:,}")
print(f"Optimizer updates: {len(history.updates):,}")
print(f"Greedy evaluation checkpoints: {len(evaluation_rows):,}")

## Prepare the recorded data

A2C now exposes the same episode-level return, progress, length, speed, and throttle summaries as REINFORCE. Update rows add actor and critic losses, weight norms, learned exploration scales, advantage statistics, and critic explained variance.

In [ ]:
episode_rows = []
for record in history.episodes:
    row = asdict(record)
    row["outcome"] = record.outcome.value
    episode_rows.append(row)
update_rows = [asdict(record) for record in history.updates]

print("Last training episode:")
print(episode_rows[-1] if episode_rows else "No episode ended within the budget.")
print("\nLast optimizer update:")
print(update_rows[-1])
print("\nLast greedy evaluation:")
print(evaluation_rows[-1])

## Task performance

Training return includes exploration; greedy return evaluates the deterministic bounded mean policy. Keeping them in one category but separate panels avoids conflating behaviour-policy noise with learned deterministic performance.

In [ ]:
plot_task_performance(
    episode_rows,
    evaluation_rows,
    moving_average_window=MOVING_AVERAGE_WINDOW,
)
plt.show()

## Progress and sample efficiency

Training and greedy maximum progress are shown with their own moving averages. Episode length provides the complementary interaction cost of a training episode.

In [ ]:
plot_progress_and_efficiency(
    episode_rows,
    evaluation_rows,
    moving_average_window=MOVING_AVERAGE_WINDOW,
)
plt.show()

## Driving behaviour

Mean speed and throttle magnitude expose stalled, timid, or excessively aggressive policies even when shaped return alone is hard to interpret.

In [ ]:
plot_driving_behavior(
    episode_rows,
    moving_average_window=MOVING_AVERAGE_WINDOW,
)
plt.show()

## Optimization and learned exploration

Actor and critic losses are separated because their scales and targets differ. Actor and critic weight norms share a panel. The throttle and steering sigma curves are the learned Gaussian exploration trace; no external exploration schedule is imposed.

In [ ]:
plot_optimization_and_exploration(
    update_rows,
    algorithm_name="A2C",
    moving_average_window=MOVING_AVERAGE_WINDOW,
    include_critic=True,
)
plt.show()

## A2C-specific diagnostics

Raw advantage dispersion shows the scale of the GAE signal before actor standardization. Explained variance indicates how much of the critic-target variation is captured by the value network.

In [ ]:
plot_algorithm_diagnostics(
    update_rows,
    algorithm_name="A2C",
)
plt.show()

## Policy after training

The final dashboard reuses the initial circuit, reset seed, deterministic action rule, and frozen normalizer, while allowing all 1,000 actions.

In [ ]:
final_rollout = watch_deterministic_rollout(
    "A2C policy after training",
    reference_track,
    agent,
    normalizer,
    ENVIRONMENT_CONFIG,
    max_steps=FINAL_RENDER_STEPS,
    reset_seed=RENDER_RESET_SEED,
    viewer_mode=VIEWER_MODE,
    frame_delay=LIVE_FRAME_DELAY,
)
print("Before training:", initial_rollout)
print("After training: ", final_rollout)